# Musanga V4 — NB07 v4_002 | 100 m ROI interior + complete-sensor RGB+MS inference

**Operational acceptance rule (objective):** produce predictions ONLY for pixels that lie at least **100 metres inside the ROI boundary** and have valid RGB, DSM and complete **same-flight** four-band MS (red, green, red-edge, NIR). Other pixels are **NoData**, never interpreted as zero Musanga.

No subjective sharpness, seams, or spectral-anomaly thresholds; no manual approval flag. DSM availability is required even though the selected model is **RGB+MS (7 channels)**. The three fold-trained RGB+MS models are ensembled for exploratory full-mosaic mapping. Their ensemble has **not** been separately held-out validated.

**Sequence:** 00 setup → 01 objective coverage audit on CPU → 02 reference footprint coverage → 03 deterministic acceptance check → 04 GPU inference → 05 example atlas. Start with **Baego / 64 tiles** and visually verify rasters; then set `SITE='ALL'` and `MAX_TILES=None` for full coverage.

The 100 m setback is measured geometrically on the ROI polygon in EPSG:32635, **not** as distance to a dataset's NoData edge or RGB mosaic boundary.

## 00 — Input registry and reproducible objective inference settings

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely scipy pandas matplotlib segmentation-models-pytorch
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict
import os,json,hashlib,gc,warnings
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window,from_bounds,transform as window_transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.features import rasterize
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
from shapely.geometry import box
from shapely.ops import unary_union
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None
DATA=ROOT/'Data';V4=ROOT/'Outputs/v4';AUDIT=V4/'00_input_audit'
OLD=ROOT/'Outputs/V3_1'; NB05=V4/'05_rgb_dsm_ms_matched_v1'
MODEL_DIR=NB05/'04_fusion_models'
OUT=V4/'07_RGB_MS_100m_ROI_complete_sensors_v2'
QA_DIR=OUT/'quality_maps'; TILE_DIR=OUT/'prediction_tiles';FIG=OUT/'figures'
for p in (QA_DIR,TILE_DIR,FIG):p.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
TRAIN=json.loads((V4/'03_paired_rgb_unet_v1/03_training_contract.json').read_text())
REG=pd.read_csv(NB05/'05_harmonized_sensor_registry.csv')
assert {'group','band','path','data_band'}.issubset(REG.columns)
MOSAICS={}
for v in V31['mosaics']:
 if str(v['file']).endswith('.tif'):
  p=Path(v['path'])
  if not p.is_file():p=DATA/'Ortho_RGB'/v['file']
  assert p.is_file(),p
  MOSAICS[v['file']]=p
assert len(MOSAICS)==6
# The exact same-flight MS registry built by NB05 (including Yoko P1/P2).
GROUP_MOSAIC={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in REG.group.unique():
 if str(group).startswith('Yoko_Carlier2020_MicaSense_'):
  yoko=[m for m in MOSAICS if 'yoko' in m.lower()]
  assert len(yoko)==1,yoko
  GROUP_MOSAIC[group]=yoko[0]
assert set(REG.group.unique())==set(GROUP_MOSAIC),(
 'MS registry changed since NB05',set(REG.group.unique())^set(GROUP_MOSAIC))
ROI=gpd.read_file(Path(V31['roi'])).to_crs('EPSG:32635')
assert ROI.crs.is_projected
ROI_FULL=unary_union(ROI.geometry)
assert ROI_FULL.is_valid and not ROI_FULL.is_empty
SITE='Baego'   # 'Baego', 'MOS', 'Yangambi', 'Yoko', 'ALL'
ROI_SETBACK_M=100.0
ROI_INTERIOR=ROI_FULL.buffer(-ROI_SETBACK_M,join_style=1)
assert not ROI_INTERIOR.is_empty,'100 m inward buffer removed the complete ROI'
QA_RES_M=2.0
MAX_TILES=64    # First GPU test. Set None after checking outputs
TILE=256
MIN_TILE_QA_FRACTION=0.01  # Efficiency only; not a quality threshold
PROB_THRESHOLD=.5
assert SITE in ('Baego','MOS','Yangambi','Yoko','ALL')
def site_of(m):
 return ('Baego' if m.startswith('Baego') else 'MOS' if m.startswith('Mos_')
         else 'Yangambi' if m.startswith('Yangambi') else 'Yoko')
SELECTED=[m for m in MOSAICS if SITE=='ALL' or site_of(m)==SITE]
def bands_for(group):
 q=REG[REG.group.eq(group)]
 assert len(q)==4 and set(q.band)=={'red','green','red_edge','nir'},group
 return {b:(Path(q[q.band.eq(b)].iloc[0].path),
            int(q[q.band.eq(b)].iloc[0].data_band))
         for b in ('red','green','red_edge','nir')}
for group in GROUP_MOSAIC:
 for p,_ in bands_for(group).values():assert p.is_file(),p
PAIRS=pd.read_csv(AUDIT/'00_rgb_dsm_pair_audit.csv')
DSM_BY_MOSAIC={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
assert set(MOSAICS)==set(DSM_BY_MOSAIC)
assert all(p.is_file() for p in DSM_BY_MOSAIC.values())
print('Mosaics:',SELECTED,'| ROI setback:',ROI_SETBACK_M,'m | outputs:',OUT)


## 01 — Objective coverage audit: ROI interior, RGB, DSM and four MS bands (CPU)

Produce a coarse **coverage-map preview** (2 m grid) solely for diagnostics and tile discovery. It does not replace the fine-resolution checks in Section 04. Acceptable pixels must meet *all* conditions; multiple flights may contribute different areas, but bands cannot be mixed between flights.

In [ ]:
from rasterio.warp import reproject
def grid_setup(mosaic):
 with rasterio.open(MOSAICS[mosaic]) as rgb:
  assert rgb.crs is not None
  interior=gpd.GeoSeries([ROI_INTERIOR],crs='EPSG:32635').to_crs(rgb.crs).iloc[0]
  geom=interior.intersection(box(*rgb.bounds))
  if geom.is_empty:return None
  xmin,ymin,xmax,ymax=geom.bounds
  left=np.floor(xmin/QA_RES_M)*QA_RES_M
  top=np.ceil(ymax/QA_RES_M)*QA_RES_M
  width=int(np.ceil((xmax-left)/QA_RES_M))
  height=int(np.ceil((top-ymin)/QA_RES_M))
  assert 0<width*height<18_000_000,('Increase QA_RES_M for this mosaic',mosaic,height,width)
  tr=Affine(QA_RES_M,0,left,0,-QA_RES_M,top)
  roi_mask=rasterize([(geom,1)],out_shape=(height,width),
                      transform=tr,fill=0,dtype='uint8').astype(bool)
  return (height,width),tr,rgb.crs,roi_mask
def read_grid_band(path,idx,shape,tr,crs,positive=False):
 with rasterio.open(path) as src:
  with WarpedVRT(src,crs=crs,transform=tr,width=shape[1],height=shape[0],
                 resampling=Resampling.bilinear) as vrt:
   a=vrt.read(idx,masked=True,out_dtype='float32')
   v=np.asarray(a.filled(np.nan),np.float32)
   valid=(~np.ma.getmaskarray(a))&np.isfinite(v)
   if positive:valid&=v>0
   return valid
def source_masks(group,shape,tr,crs):
 masks=[read_grid_band(p,i,shape,tr,crs,positive=True)
          for p,i in bands_for(group).values()]
 return np.logical_and.reduce(masks)
def audit_one(mosaic):
 setup=grid_setup(mosaic)
 if setup is None:
  return dict(mosaic=mosaic,flight='',qa_path='',
              roi_interior_pixels=0,accepted_fraction=0.)
 shape,tr,crs,roi_mask=setup
 with rasterio.open(MOSAICS[mosaic]) as rgb:
  valid_rgb=np.logical_and.reduce([
      read_grid_band(MOSAICS[mosaic],i,shape,tr,crs) for i in (1,2,3)])
  # RGB alpha means transparency, not a fourth RGB reflectance channel.
  if rgb.count>=4 and rgb.colorinterp[3].name=='alpha':
   valid_rgb&=read_grid_band(MOSAICS[mosaic],4,shape,tr,crs,positive=True)
 valid_dsm=read_grid_band(DSM_BY_MOSAIC[mosaic],1,shape,tr,crs)
 ms_masks={}
 for group,owner in GROUP_MOSAIC.items():
  if owner==mosaic:ms_masks[group]=source_masks(group,shape,tr,crs)
 assert len(ms_masks)
 any_ms=np.logical_or.reduce(list(ms_masks.values()))
 good=roi_mask&valid_rgb&valid_dsm&any_ms
 # 1=inside ROI interior and RGB valid; 2=RGB+DSM; 3=all required sensors
 status=np.zeros(shape,np.uint8)
 status[roi_mask&valid_rgb]=1
 status[roi_mask&valid_rgb&valid_dsm]=2
 status[good]=3
 path=QA_DIR/(mosaic.replace('.tif','')+'_objective_sensor_support_100m.tif')
 with rasterio.open(path,'w',driver='GTiff',height=shape[0],
         width=shape[1],count=1,dtype='uint8',crs=crs,transform=tr,
         nodata=0,compress='deflate',tiled=True) as out:out.write(status,1)
 fig,axs=plt.subplots(1,3,figsize=(14,4),layout='constrained')
 for ax,arr,title in zip(axs,
         [roi_mask,any_ms,status],
         ['ROI interior (100 m setback)','Any complete 4-band MS flight',
          'Full sensor support (3 = RGB+DSM+MS)']):
  ax.imshow(arr,cmap='viridis',interpolation='nearest')
  ax.set_title(title);ax.axis('off')
 fig.suptitle(f'{mosaic}: {good.sum()/max(roi_mask.sum(),1):.1%} with all sensors inside ROI interior')
 fig.savefig(FIG/(mosaic.replace('.tif','')+'_objective_sensor_support.png'),dpi=170)
 plt.show();plt.close(fig)
 return dict(mosaic=mosaic,flight='per-tile selection',qa_path=str(path),
      roi_interior_pixels=int(roi_mask.sum()),
      rgb_available_fraction=float((valid_rgb&roi_mask).sum()/max(roi_mask.sum(),1)),
      dsm_available_fraction=float((valid_dsm&roi_mask).sum()/max(roi_mask.sum(),1)),
      ms_available_fraction=float((any_ms&roi_mask).sum()/max(roi_mask.sum(),1)),
      accepted_fraction=float(good.sum()/max(roi_mask.sum(),1)))
QA=pd.DataFrame([audit_one(m) for m in SELECTED])
QA.to_csv(OUT/'01_objective_sensor_coverage.csv',index=False)
display(QA.round(3))
print('This is a diagnostic at 2 m. Inference repeats every validity test at native RGB resolution.')


## 02 — How many held-out reference footprints remain within the objective inference domain?

In [ ]:
UNITS=gpd.read_file(V4/'01_spatial_cover_folds_v2/01_reference_groups_and_folds_v2.gpkg')
UNITS.unit_id=UNITS.unit_id.astype(str)
OWNERS=pd.read_csv(V4/'02_rgb_circles_plus_plots_v3/02_reference_mosaic_owners_v2.csv')
OWNERS.unit_id=OWNERS.unit_id.astype(str)
UNITS=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
 on=['unit_type','unit_id'],validate='one_to_one')
UNITS=UNITS[UNITS.roi_mosaic_coverage_fraction.ge(.95)]
assert len(UNITS)==106
coverage=[]
for mosaic in SELECTED:
 qa_file=Path(QA.set_index('mosaic').loc[mosaic,'qa_path'])
 if not qa_file.is_file():continue
 with rasterio.open(qa_file) as ds:
  status=ds.read(1)
  for z in UNITS[UNITS.mosaic.eq(mosaic)].to_crs(ds.crs).itertuples():
   footprint=rasterize([(z.geometry,1)],out_shape=status.shape,
     transform=ds.transform,fill=0,dtype='uint8').astype(bool)
   if not footprint.any():continue
   coverage.append(dict(site=z.site,unit_type=z.unit_type,unit_id=z.unit_id,
      mosaic=mosaic,all_sensor_fraction=float((status[footprint]==3).mean()),
      ROI_or_RGB_unavailable=float((status[footprint]==0).mean()),
      RGB_only_fraction=float((status[footprint]==1).mean()),
      RGB_DSM_without_MS=float((status[footprint]==2).mean())))
REFERENCE_QA=pd.DataFrame(coverage)
REFERENCE_QA.to_csv(OUT/'02_reference_quality_by_footprint.csv',index=False)
print('Lowest objective all-sensor coverage in the 100 m ROI interior:')
display(REFERENCE_QA.sort_values('all_sensor_fraction').head(20).round(3))


## 03 — Deterministic readiness check (no manual quality approval)

Automatic inference is permitted only on pixels inside the inward-buffered ROI and with all sensor measurements available. No visual quality classification is used.

In [ ]:
assert len(QA)==len(SELECTED)
assert all(Path(v).is_file() for v in QA.qa_path if isinstance(v,str) and v)
assert QA.roi_interior_pixels.ge(0).all()
assert QA.accepted_fraction.between(0,1).all()
print('Objective ROI setback:',ROI_SETBACK_M,'metres')
print('Ready for GPU inference. Only complete RGB + DSM + 4-band MS pixels will be written.')
display(QA[['mosaic','accepted_fraction']])


## 04 — RGB+MS ensemble inference on approved tiles only (GPU)

In [ ]:
import torch
import segmentation_models_pytorch as smp
assert QA_APPROVED is True,'Quality approval required'
assert torch.cuda.is_available(),'Switch Colab runtime to GPU, rerun Section 00 and approval gate'
DEVICE='cuda'
def rgb_model_A(fold):
 nb03=V4/'03_paired_rgb_unet_v1'
 ck=torch.load(nb03/f'fold{fold}_A_{TRAIN["training_key"][:18]}.pt',
   map_location='cpu',weights_only=False)
 assert ck['training_key']==TRAIN['training_key']
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
 net.load_state_dict({k.removeprefix('net.'):v for k,v in ck['model'].items()})
 return net
def ms_net(fold):
 base=rgb_model_A(fold).state_dict()
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=7,classes=1)
 state=net.state_dict()
 for k,v in state.items():
  if k=='encoder.conv1.weight':
   v[:,:3]=base[k];v[:,3:]=0
  else:v.copy_(base[k])
 net.load_state_dict(state)
 ck=torch.load(MODEL_DIR/f'fold{fold}_RGB_MS_v1.pt',
       map_location='cpu',weights_only=False)
 assert ck['complete'] and ck['baseline_training_key']==TRAIN['training_key']
 net.load_state_dict(ck['state'],strict=True)
 return net.to(DEVICE).eval(),ck['fingerprint']
MODELS=[];MODEL_HASH=[]
for fold in range(3):
 model,h=ms_net(fold);MODELS.append(model);MODEL_HASH.append(h)
MODEL_SIGNATURE=hashlib.sha256(json.dumps(MODEL_HASH).encode()).hexdigest()[:12]
QA_SIGNATURE=hashlib.sha256(json.dumps(dict(
    resolution=QA_RES_M,edge=EDGE_BUFFER_M,dev=MAX_SPECTRAL_DEVIATION,
    neighbor=NEIGHBORHOOD_M,suspect=MAX_ANOMALY_FRACTION,
    exclude=str(MANUAL_BAD_POLYGONS),override=FLIGHT_OVERRIDE,
    qa_files=[(str(x.qa_path),Path(x.qa_path).stat().st_mtime_ns)
              for x in QA.itertuples()]),sort_keys=True).encode()).hexdigest()[:12]
print('Model signature',MODEL_SIGNATURE,'QA signature',QA_SIGNATURE)
def make_vrts(ds,group,stack):
 result={}
 for b,(p,index) in bands_for(group).items():
  src=stack.enter_context(rasterio.open(p))
  vrt=stack.enter_context(WarpedVRT(src,crs=ds.crs,transform=ds.transform,
       width=ds.width,height=ds.height,resampling=Resampling.bilinear))
  result[b]=(vrt,index)
 return result
def read_ms_window(vrts,win):
 layers=[];valid=[]
 for b in ('red','green','red_edge','nir'):
  vrt,index=vrts[b];a=vrt.read(index,window=win,masked=True,out_dtype='float32')
  v=np.asarray(a.filled(0),np.float32)
  ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
  layers.append(np.where(ok,v,0));valid.append(ok)
 raw=np.stack(layers);valid=np.logical_and.reduce(valid)
 out=np.where(valid[None],raw/np.maximum(raw.sum(axis=0,keepdims=True),1e-8),0.)
 return out.astype('float32'),valid
def rgb_window(ds,win):
 a=ds.read([1,2,3],window=win,masked=True)
 assert all(np.dtype(t)==np.dtype('uint8') for t in ds.dtypes[:3])
 ok=~np.ma.getmaskarray(a).any(axis=0)
 if ds.count>=4:
  alpha=ds.read(4,window=win,masked=True)
  ok &= (~np.ma.getmaskarray(alpha))&(np.asarray(alpha.filled(0))>0)
 x=np.asarray(a.filled(0),np.float32)/255.
 return x,ok
def qa_window(src,win,ds):
 out=np.zeros((int(win.height),int(win.width)),np.uint8)
 reproject(source=rasterio.band(src,1),destination=out,
           src_transform=src.transform,src_crs=src.crs,
           dst_transform=ds.window_transform(win),dst_crs=ds.crs,
           src_nodata=0,dst_nodata=0,resampling=Resampling.nearest)
 return out==3
def tiles_for_qa(ds,qa):
 # Enumerate only QA-good 2m pixels and convert their bounding raster range;
 # no full-sized RGB allocation, then one 256px tile at a time.
 status=qa.read(1)
 good_y,good_x=np.where(status==3)
 if not len(good_x):return []
 xy=[qa.xy(int(good_y.min()),int(good_x.min())),
     qa.xy(int(good_y.max()),int(good_x.max()))]
 rowcols=[ds.index(x,y) for x,y in xy]
 r0=max(0,min(q[0] for q in rowcols)//TILE)
 r1=min((ds.height-1)//TILE,max(q[0] for q in rowcols)//TILE)
 c0=max(0,min(q[1] for q in rowcols)//TILE)
 c1=min((ds.width-1)//TILE,max(q[1] for q in rowcols)//TILE)
 return [(r,c) for r in range(r0,r1+1) for c in range(c0,c1+1)]
@torch.inference_mode()
def run_mosaic(mosaic):
 qa_rec=QA.set_index('mosaic').loc[mosaic]
 group=qa_rec.flight
 mdir=TILE_DIR/mosaic.replace('.tif','')
 mdir.mkdir(parents=True,exist_ok=True)
 manifest=[]
 saved=0
 with ExitStack() as stack:
  rgb=stack.enter_context(rasterio.open(MOSAICS[mosaic]))
  qa=stack.enter_context(rasterio.open(qa_rec.qa_path))
  vrt=make_vrts(rgb,group,stack)
  locations=tiles_for_qa(rgb,qa)
  for row,col in locations:
   h=min(TILE,rgb.height-row*TILE);w=min(TILE,rgb.width-col*TILE)
   win=Window(col*TILE,row*TILE,w,h)
   qmask=qa_window(qa,win,rgb)
   good_frac=float(qmask.mean())
   if good_frac<MIN_TILE_QA_FRACTION:continue
   dest=mdir/f'qa{QA_SIGNATURE}_m{MODEL_SIGNATURE}_r{row:05d}_c{col:05d}.tif'
   if dest.is_file():
    manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(dest),status='reused',good_fraction=good_frac))
    saved+=1
    if MAX_TILES is not None and saved>=MAX_TILES:break
    continue
   x,rgbok=rgb_window(rgb,win)
   ms,msok=read_ms_window(vrt,win)
   valid=qmask&rgbok&msok
   if not valid.any():continue
   features=np.zeros((7,TILE,TILE),np.float32)
   features[:3,:h,:w]=x
   features[3:,:h,:w]=ms
   tensor=torch.from_numpy(features[None]).to(DEVICE)
   probs=[]
   for model in MODELS:
    with torch.autocast('cuda'):
     probs.append(torch.sigmoid(model(tensor)[:,0]).float()[0].cpu().numpy()[:h,:w])
   probability=np.mean(probs,axis=0).astype('float32')
   probability[~valid]=-9999.
   profile=rgb.profile.copy()
   profile.update(driver='GTiff',count=2,height=h,width=w,
        transform=rgb.window_transform(win),dtype='float32',
        nodata=-9999.,compress='deflate',tiled=False)
   tmp=dest.with_name(dest.stem+'.temp.tif')
   binary=np.where(valid,(probability>=PROB_THRESHOLD).astype('float32'),-9999.)
   with rasterio.open(tmp,'w',**profile) as dst:
    dst.write(probability,1);dst.write(binary,2)
    dst.set_band_description(1,'Musanga probability (approved MS only)')
    dst.set_band_description(2,'Musanga binary prediction (approved MS only)')
   os.replace(tmp,dest)
   manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(dest),status='new',
                        good_fraction=good_frac,valid_fraction=float(valid.mean())))
   saved+=1
   if saved%10==0:print('Saved',mosaic,saved,flush=True)
   if MAX_TILES is not None and saved>=MAX_TILES:break
 return manifest
TILE_MANIFEST=[]
for name in SELECTED:TILE_MANIFEST+=run_mosaic(name)
TILE_MANIFEST=pd.DataFrame(TILE_MANIFEST)
TILE_MANIFEST.to_csv(OUT/'04_prediction_tile_manifest.csv',index=False)
print('Saved/reused inference tiles:',len(TILE_MANIFEST))
display(TILE_MANIFEST.groupby(['mosaic','status']).size().rename('tiles').reset_index()
        if len(TILE_MANIFEST) else TILE_MANIFEST)


## 05 — Circle/plot example atlas: RGB, MS, annotations and predictions (from actual inferred tiles)

In [ ]:
import matplotlib.colors as mcolors
from rasterio.merge import merge
# To keep figures honest, use only units actually intersecting saved prediction
# tiles and require >=50% quality-approved support within the shown unit.
assert 'TILE_MANIFEST' in globals() and len(TILE_MANIFEST)>0, 'Run Section 04 first. No accepted inference tiles exist.'
PLOT_SOURCE=gpd.read_file(F78['exhaustive_plot_crowns']).to_crs('EPSG:32635')
PLOT_SOURCE['positive']=pd.to_numeric(PLOT_SOURCE.Class_ID,errors='coerce').eq(1)
CIRCLE_SOURCE=gpd.read_file(F78['circle_positive_crowns']).to_crs('EPSG:32635')
CIRCLE_SOURCE['positive']=True
def plot_a_few():
 used=[]
 for mosaic,files in TILE_MANIFEST.groupby('mosaic'):
  units=UNITS[UNITS.mosaic.eq(mosaic)].copy()
  if not len(units):continue
  tilefiles=list(files.path)
  # Reference footprints can straddle tile boundaries. Select one footprint
  # whose whole buffered bounding box intersects saved tiles; mosaic merge
  # below reconstructs an excerpt from the georeferenced probability tiles.
  with ExitStack() as stack:
   src=[stack.enter_context(rasterio.open(p)) for p in tilefiles]
   union_bounds=unary_union([box(*p.bounds) for p in src])
   with rasterio.open(MOSAICS[mosaic]) as rgb:
    with ExitStack() as vst:
     group=QA.set_index('mosaic').loc[mosaic,'flight']
     vrts=make_vrts(rgb,group,vst)
     for z in units.to_crs(rgb.crs).itertuples():
      if not union_bounds.covers(z.geometry):continue
      win=from_bounds(*z.geometry.bounds,transform=rgb.transform)
      c0=max(0,int(np.floor(win.col_off)));r0=max(0,int(np.floor(win.row_off)))
      c1=min(rgb.width,int(np.ceil(win.col_off+win.width)))
      r1=min(rgb.height,int(np.ceil(win.row_off+win.height)))
      ww=c1-c0;hh=r1-r0
      if ww<15 or hh<15 or ww*hh>8_000_000:continue
      window=Window(c0,r0,ww,hh)
      ptiles=[q for q in src if box(*q.bounds).intersects(z.geometry)]
      if not ptiles:continue
      pred,tr=merge(ptiles,bounds=rasterio.windows.bounds(window,rgb.transform),
                    nodata=-9999.,indexes=[1],res=(abs(rgb.transform.a),abs(rgb.transform.e)))
      p=np.asarray(pred[0],float)
      if p.shape!=(hh,ww):continue
      refmask=rasterize([(z.geometry,1)],out_shape=(hh,ww),
               transform=rgb.window_transform(window),fill=0).astype(bool)
      good=(p>=0)&(p<=1)
      if not refmask.any() or good[refmask].mean()<.5:continue
      rgb3,rv=rgb_window(rgb,window)
      ms,msv=read_ms_window(vrts,window)
      false=np.stack([ms[3],ms[2],ms[0]],axis=-1)
      validMS=msv&(ms.sum(axis=0)>0)
      if validMS.any():
       for band in range(3):
        q=false[:,:,band]
        lo,hi=np.percentile(q[validMS],[2,98])
        false[:,:,band]=np.clip((q-lo)/max(hi-lo,1e-8),0,1)
      false[~validMS]=1
      extent=rasterio.windows.bounds(window,rgb.transform)
      extent=[extent[0],extent[2],extent[1],extent[3]]
      fig,axes=plt.subplots(1,6,figsize=(21,3.7),layout='constrained')
      axes[0].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      axes[1].imshow(false,extent=extent,origin='upper')
      axes[2].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      src_gdf=CIRCLE_SOURCE if z.unit_type=='circle' else PLOT_SOURCE
      selected=src_gdf.iloc[src_gdf.sindex.query(
          gpd.GeoSeries([z.geometry],crs=rgb.crs).to_crs('EPSG:32635').iloc[0],
          predicate='intersects')].copy()
      if len(selected):
       selected=selected.to_crs(rgb.crs)
       for positive,color in [(True,'#ff5544'),(False,'#39c8ed')]:
        part=selected[selected.positive.eq(positive)]
        if len(part):part.boundary.plot(ax=axes[2],color=color,linewidth=.8)
      probability=np.ma.masked_where(~good,p)
      axes[3].imshow(probability,extent=extent,origin='upper',cmap='magma',vmin=0,vmax=1)
      axes[4].imshow(np.ma.masked_where(~good,p>=PROB_THRESHOLD),
                    extent=extent,origin='upper',cmap='Greens',vmin=0,vmax=1)
      axes[5].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      overlay=np.ma.masked_where(~good|(p<PROB_THRESHOLD),p)
      axes[5].imshow(overlay,extent=extent,origin='upper',cmap='spring',vmin=0,vmax=1,alpha=.50)
      for ax,title in zip(axes,['RGB','MS NIR/RE/R','Annotations','Probability',
                                 'Binary Musanga','Prediction overlay']):
       gpd.GeoSeries([z.geometry],crs=rgb.crs).boundary.plot(
          ax=ax,color='#ffec4d',linewidth=1.4)
       ax.set(xlim=(extent[0],extent[1]),ylim=(extent[2],extent[3]),title=title)
       ax.set_xticks([]);ax.set_yticks([])
      fig.suptitle(f'{z.site} {z.unit_type} {z.unit_id} | mapped MS quality only')
      out=FIG/f'example_{mosaic.replace(".tif","")}_{z.unit_type}_{z.unit_id}.png'
      fig.savefig(out,dpi=175,bbox_inches='tight');plt.show();plt.close(fig)
      used.append(dict(site=z.site,unit_type=z.unit_type,unit_id=z.unit_id,
                       path=str(out),approved_fraction=float(good[refmask].mean())))
      if len(used)>=6:return used
 return used
EXAMPLES=pd.DataFrame(plot_a_few())
EXAMPLES.to_csv(OUT/'05_example_atlas_manifest.csv',index=False)
print('Completed QA example panels:',len(EXAMPLES))
display(EXAMPLES)
if not len(EXAMPLES):print('Smoke-test tiles did not fully cover any reference footprint. '
                            'Expand MAX_TILES or choose a reference-centered tile set.')
